# B2-024-gpu-scientific-ml-capstone — Practice p05 — Solution

**Type:** mc · **Difficulty:** core · **Concepts:** mixture-parameter-regression

*20 minutes.*  
**Set:** A  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Solution

We need a loss $L(\hat P,T)$ with two properties:
(I) **permutation-invariance**: $L(\sigma\hat P,T)=L(\hat P,T)=L(\hat P,\tau T)$ for all component relabellings $\sigma,\tau$; and
(Z) **exact zero set**: $L(\hat P,T)=0 \iff \hat P=\pi T$ for some relabelling $\pi$.

**A (fixed-order squared error).** Not invariant. Counterexample with two components: $T=\big((1,0.2,0.05),(1,0.7,0.05)\big)$ and $\hat P$ = the same two triples in the other order. Then $\hat P$ equals $T$ up to relabelling but the loss is $\tfrac{2\cdot0.5^2}{6}>0$. (Its zero set is "equal in this exact order", which is too small.)

**B (minimum over the six permutations of the mean absolute difference).** Invariant: relabelling $\hat P$ by $\sigma$ just re-indexes the set $\{\pi\}$ over which the minimum is taken ($\pi\mapsto\pi\circ\sigma$ is a bijection of the six permutations), and relabelling $T$ likewise. Zero set: the minimum of finitely many non-negative numbers is $0$ iff one of them is $0$, i.e. iff $|\hat P_{\pi(k),j}-T_{k,j}|=0$ for all $k,j$ for some $\pi$, i.e. iff $\hat P$ equals $T$ up to relabelling. **Both properties hold.**

**C (sort $\hat P$ by weight, $T$ by center).** Each side is put in a canonical order, so it is invariant (when there are no ties). But the zero set is wrong: take $T=\big((2,0.2,0.05),(1,0.7,0.05)\big)$, already a valid ordering, and $\hat P=T$. Sorted by center $T$ stays $\big((2,0.2,\cdot),(1,0.7,\cdot)\big)$; sorted by weight $\hat P$ becomes $\big((1,0.7,\cdot),(2,0.2,\cdot)\big)$; the loss is positive although $\hat P=T$ exactly.

**D (squared differences of the sums).** Invariant (sums do not depend on order). Zero set too large: $T=\big((1,0.25,0.05),(1,0.75,0.05)\big)$ and $\hat P=\big((1,0.375,0.05),(1,0.625,0.05)\big)$ have equal weight, center and width sums, so the loss is $0$, yet no relabelling maps one to the other.

**E (sort all nine entries as one list).** Invariant (the multiset of entries does not depend on order). Zero set too large: $T=\big((1,0.2,0.05),(0.5,0.7,0.06)\big)$ and $\hat P=\big((1,0.7,0.05),(0.5,0.2,0.06)\big)$ (the two centers swapped between components) contain the same multiset of entries, so the loss is $0$, but $\hat P$ is not a relabelling of $T$.

**Answer: B.** It is exactly `capstone_data.perm_invariant_error_per_example`, the unit's mixture metric.

The cell below checks every counterexample numerically (with two components, so "six permutations" becomes two).

In [ ]:
import itertools
import torch


def loss_A(p, t):
    return ((p - t) ** 2).mean().item()


def loss_B(p, t):
    k = p.shape[0]
    return min((p[list(perm)] - t).abs().mean().item() for perm in itertools.permutations(range(k)))


def loss_C(p, t):
    ps = p[torch.argsort(p[:, 0], stable=True)]
    ts = t[torch.argsort(t[:, 1], stable=True)]
    return (ps - ts).abs().mean().item()


def loss_D(p, t):
    return ((p.sum(0) - t.sum(0)) ** 2).sum().item()


def loss_E(p, t):
    return (p.flatten().sort().values - t.flatten().sort().values).abs().mean().item()


f64 = lambda rows: torch.tensor(rows, dtype=torch.float64)
T_a = f64([[1, 0.2, 0.05], [1, 0.7, 0.05]])
swap = [1, 0]
# A: not invariant
a_swapped = loss_A(T_a[swap], T_a)
# B: invariant and zero exactly on relabellings
b_same = loss_B(T_a[swap], T_a)
T_d = f64([[1, 0.25, 0.05], [1, 0.75, 0.05]])
P_d = f64([[1, 0.375, 0.05], [1, 0.625, 0.05]])   # dyadic centers: sums are exact
b_other = loss_B(P_d, T_d)
# C: wrong zero set
T_c = f64([[2, 0.2, 0.05], [1, 0.7, 0.05]])
c_equal = loss_C(T_c.clone(), T_c)
# D: zero for a non-relabelling
d_fake = loss_D(P_d, T_d)
# E: zero for a non-relabelling
T_e = f64([[1, 0.2, 0.05], [0.5, 0.7, 0.06]])
P_e = f64([[1, 0.7, 0.05], [0.5, 0.2, 0.06]])
e_fake = loss_E(P_e, T_e)
b_e = loss_B(P_e, T_e)
ANSWER = "B"
print(dict(a_swapped=a_swapped, b_same=b_same, b_other=b_other, c_equal=c_equal, d_fake=d_fake, e_fake=e_fake, b_e=b_e))

### Answer check

In [ ]:
assert a_swapped > 0                      # A: a relabelling of T has positive loss
assert b_same == 0 and b_other > 0 and b_e > 0
assert c_equal > 0                        # C: positive loss although P == T
assert d_fake == 0 and loss_B(P_d, T_d) > 0   # D: zero loss for a different mixture
assert e_fake == 0                        # E: zero loss for a different mixture
assert ANSWER == "B"